In [1]:
import pandas as pd
import sqlite3
from scipy import stats

In [2]:
df = pd.read_csv("data.csv")

print(df)

   Customer_ID    Name    Region Join_Month Website_Version  Spend_Amount  \
0          101    Aman  Region_A    2026-01      Old_Layout          4500   
1          102   Priya  Region_A    2026-01      New_Layout          5500   
2          103   Rahul  Region_B    2026-02      Old_Layout          3000   
3          104   Sneha  Region_B    2026-02      Old_Layout          3200   
4          105    Amit  Region_A    2026-02      New_Layout          6500   
5          106    Riya  Region_B    2026-03      New_Layout          4200   
6          107  Vikram  Region_A    2026-03      Old_Layout          4800   
7          108    Neha  Region_B    2026-03      New_Layout          5000   
8          109  Sanjay  Region_A    2026-03      New_Layout          5800   
9          110   Pooja  Region_B    2026-03      Old_Layout          3500   

   Satisfaction_Score  
0                  65  
1                  88  
2                  30  
3                  45  
4                  92  
5       

In [3]:
print(df.dtypes)

Customer_ID           int64
Name                    str
Region                  str
Join_Month              str
Website_Version         str
Spend_Amount          int64
Satisfaction_Score    int64
dtype: object


In [4]:
print(df.isnull().sum())

Customer_ID           0
Name                  0
Region                0
Join_Month            0
Website_Version       0
Spend_Amount          0
Satisfaction_Score    0
dtype: int64


In [5]:
print(df["Spend_Amount"].mean())

4600.0


In [6]:
print(df["Spend_Amount"].median())

4650.0


In [7]:
print(df["Spend_Amount"].skew())

0.10444483721129422


In [8]:
pivot = df.pivot_table(
    values="Satisfaction_Score",
    index="Region",
    columns="Website_Version",
    aggfunc="mean"
)

print(pivot)

Website_Version  New_Layout  Old_Layout
Region                                 
Region_A          86.666667        67.5
Region_B          30.000000        45.0


In [9]:
old = df[df["Website_Version"] == "Old_Layout"]["Spend_Amount"]

new = df[df["Website_Version"] == "New_Layout"]["Spend_Amount"]

In [10]:
from scipy import stats

t_stat, p_value = stats.ttest_ind(old, new, equal_var=False)

print("T-statistic:", t_stat)
print("P-value:", p_value)

T-statistic: -3.034572967243526
P-value: 0.016294713858702244


In [11]:
if p_value < 0.05:
    print("Reject the Null Hypothesis")
else:
    print("Fail to Reject the Null Hypothesis")

Reject the Null Hypothesis


In [12]:
import sqlite3

conn = sqlite3.connect("bazaar_assignment.db")

In [13]:
df.to_sql("bazaar_ledger", conn, if_exists="replace", index=False)

10

In [14]:
query = """
SELECT Customer_ID,
       Name,
       Spend_Amount
FROM bazaar_ledger
WHERE Spend_Amount >= 4000;
"""

result = pd.read_sql_query(query, conn)

print(result)

   Customer_ID    Name  Spend_Amount
0          101    Aman          4500
1          102   Priya          5500
2          105    Amit          6500
3          106    Riya          4200
4          107  Vikram          4800
5          108    Neha          5000
6          109  Sanjay          5800


In [15]:
query = """
SELECT Region,
       SUM(Spend_Amount) AS Total_Revenue
FROM bazaar_ledger
GROUP BY Region;
"""

result = pd.read_sql_query(query, conn)

print(result)

     Region  Total_Revenue
0  Region_A          27100
1  Region_B          18900


In [16]:
conn.close()